# Downscale training and analysis

A readable entry point for the Colab experiment. **All model, data, optimizer, and evaluation code is in `src/fox_experiments/`.** Each cell below performs one visible step.

**First run:** Colab → Runtime → Change runtime type → GPU; set `REPO_URL` after pushing the repository. Start with `PROFILE="smoke"`, then use `"downscale"`. Locally, open this notebook inside your clone.

The comparison preserves the original design: real-text acquisition → short latest-write acquisition → paired SGD / fixed Adam / annealed Adam continuation, with factorized/direct first gates and the original FoX gate. Read [the scientific design](../docs/experiment_design.md) before interpreting results.

CPU smoke results verify software only. Finite text tests cannot establish infinite generalization.

For the original FoX architecture and paper training recipe, use [03_paper_baseline_comparison.ipynb](03_paper_baseline_comparison.ipynb). Its from-initialization natural-text study is separate from the adapted AdamW control below.

## 1. Open the repository

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = ""  # After publishing: https://github.com/YOUR_NAME/fox_experiments.git
REPO_REF = "main"  # Branch, tag, or commit to use in a new Colab clone.
candidates = [Path.cwd(), Path.cwd().parent, Path("/content/fox_experiments")]
REPO = next((p for p in candidates if (p / "src/fox_experiments").is_dir()), None)
if REPO is None:
    if not REPO_URL:
        raise ValueError("Set REPO_URL above, or open this notebook inside a local clone.")
    REPO = Path("/content/fox_experiments")
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    subprocess.check_call(["git", "-C", str(REPO), "checkout", REPO_REF])
REPO = REPO.resolve()
os.chdir(REPO)
print("Repository:", REPO)
print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
      if (REPO / ".git/HEAD").exists() and subprocess.run(
          ["git", "rev-parse", "--verify", "HEAD"], capture_output=True).returncode == 0
      else "No commit yet: commit the source before a scientific run.")

## 2. Install the project and check the runtime

In [ ]:
if os.environ.get("FOX_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)])
sys.path.insert(0, str(REPO / "src"))  # Make this checkout visible to the current kernel.

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
print("PyTorch:", torch.__version__, "| Device:", DEVICE)

## 3. Choose the budget and storage

Edit the JSON file to change the model, optimizer rates, lag grid, or training steps. Create a new config/run name for a scientific change. Set `RESUME=True` only to continue the same settings and data. Google Drive is optional but recommended in Colab so results survive disconnections.

In [ ]:
from fox_experiments.training import ExperimentConfig, budget

PROFILE = os.environ.get("FOX_PROFILE", "downscale")  # smoke / downscale / replicate
USE_GOOGLE_DRIVE = False  # Set True on Colab for persistent data and checkpoints.
RESUME = False
RUN_NAME = os.environ.get("FOX_RUN_NAME", PROFILE + "_v2")
WORKSPACE = Path(os.environ.get("FOX_WORKSPACE", str(REPO / "outputs")))
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKSPACE = Path("/content/drive/MyDrive/fox_experiments")
CONFIG = ExperimentConfig.from_json(REPO / "configs" / (PROFILE + ".json"))
OUT = WORKSPACE / "text" / RUN_NAME
DATA_DIR = Path(os.environ.get("FOX_DATA_DIR", str(REPO / "data/longcrawl64_pilot")))
if USE_GOOGLE_DRIVE:
    DATA_DIR = WORKSPACE / "data/longcrawl64_pilot"
print("Output:", OUT)
print(budget(CONFIG))

In [ ]:
import pandas as pd
from IPython.display import display, Image

display(pd.Series(CONFIG.to_dict(), name="Setting").to_frame())
if DEVICE == "cpu" and PROFILE != "smoke":
    raise RuntimeError("Enable a GPU, or choose PROFILE='smoke' for a software check.")

## 4. Prepare and inspect the corpus

The local folder already contains the verified 18 MiB native subset. A fresh clone downloads about 248 MB once to reconstruct it; later runs check its hash and reuse it. Pilot splits repurpose distinct source-heldout rows and are separate from the full experiment's benchmark splits.

Downloader: `src/fox_experiments/data/download.py`. Task construction: `data/probes.py`. The model receives tokens only; target positions below are diagnostic metadata.

The data cache is unchanged. Generated v2 task examples draw an independently wrong old value; old v1 run predictions remain valid as records of that earlier protocol.

In [ ]:
from fox_experiments.cli import prepare_data

DATA, manifest = prepare_data(DATA_DIR)
print("Corpus checksum:", manifest["sha256"])
print("Split counts:", manifest["split_counts"])
example = DATA.example("confirm", seed=17, lag=min(CONFIG.train_lags),
                       base_length=CONFIG.train_length)
print(DATA.enc.decode(example["tokens"]))
print("Expected answer:", DATA.enc.decode([example["answer"]]))

## 5. Train the optimizer and gate comparisons

Implementation: `training/experiment.py`, `training/trainer.py`, and `training/optimizers.py`. Learning rates are selected using short validation only. Each gate/optimizer branch starts from its paired acquired function; long-test scores do not choose rates.

An existing run requires `RESUME=True`; a changed configuration requires a new run directory. Interrupting Colab loses work since the last saved checkpoint.

Scientific runs now stop before the optimizer grid if short acquisition fails. A zero short score is not a generalization boundary. Smoke runs deliberately retain untrained branches for software testing. The corrected v2 probe removes the deterministic old-value shortcut; use a fresh run name after updating.

In [ ]:
from fox_experiments.training import run_experiment

artifacts = run_experiment(DATA, CONFIG, OUT, device=DEVICE, resume=RESUME,
                           allow_unqualified=(PROFILE == "smoke"))
print(artifacts)

### Check acquisition and explain zeros

Inspect this table before interpreting length curves. A constant digit guess can have nonzero individual accuracy and zero all-edit accuracy. Diagnosis reads saved CSV files; model weights are not needed.

In [ ]:
from fox_experiments.evaluation import diagnose_run

quality = diagnose_run(OUT)
display(pd.read_csv(OUT / "acquisition.csv"))
if "csv" in quality:
    display(pd.read_csv(quality["csv"])[[
        "branch", "checkpoint_step", "base_accuracy", "all_edit_history_accuracy",
        "latest_edit_prediction_change_rate", "distinct_prediction_tokens", "diagnosis"]])
print("Diagnostic report:", quality["report"])

## 6. Analyze saved predictions

The plots separate **target lag** from **older stale prefixes**. All-edit history accuracy requires the correct full-vocabulary answer across the applicable edits. First inspect short acquisition; a model that never learned the short rule does not locate a generalization boundary.

A curve passing at the largest tested lag is right-censored. One seed cannot measure variability across training seeds.

In [ ]:
from fox_experiments.evaluation import summarize_results

if (OUT / "retrieval_raw.csv").exists():
    panels, ranges, contrasts, figures = summarize_results(OUT)
    display(ranges)
    display(contrasts.head(12))
    for figure in figures:
        display(Image(filename=str(figure)))
else:
    print("Short acquisition failed; no optimizer comparison was run.")

## 7. Controlled binding-gate experiment

This smaller model separately tests exactly `g=softplus(u*v)` versus `g=softplus(z)`. Its analytic diagnostics can eliminate the number of older records for this specific model. They do not certify an ordinary multilayer language model.

`R=2` is the manuscript setup; `R=4,8` change the recall objective and test an **unproved** candidate cutoff near `R+2`. Controlled `sgd` is full-batch gradient descent. This small controlled run does not resume partial training: use a fresh `RUN_NAME` after an interruption or a settings change. Completed matching runs can be re-analyzed. See `mechanism/` for the model, tasks, bounds and training code.

In [ ]:
from fox_experiments.mechanism import run_mechanism
from fox_experiments.evaluation import plot_mechanism
import json

RUN_MECHANISM = True
MECHANISM_OUT = WORKSPACE / "mechanism" / RUN_NAME
mechanism_profile = "smoke" if PROFILE == "smoke" else "pilot"
recall_lags = (2,) if PROFILE == "smoke" else (2, 4, 8)
if (MECHANISM_OUT / "summary.csv").exists():
    saved = json.loads((MECHANISM_OUT / "config.json").read_text())
    if (saved["profile"], saved["seeds"], saved["recall_lags"]) != (
            mechanism_profile, list(CONFIG.seeds), list(recall_lags)):
        raise ValueError("Controlled settings changed; choose a new RUN_NAME.")
elif RUN_MECHANISM:
    run_mechanism(MECHANISM_OUT, profile=mechanism_profile,
                  seeds=CONFIG.seeds, recall_lags=recall_lags)
if (MECHANISM_OUT / "summary.csv").exists():
    display(pd.read_csv(MECHANISM_OUT / "summary.csv"))
    for figure in plot_mechanism(MECHANISM_OUT):
        display(Image(filename=str(figure)))
if (MECHANISM_OUT / "summary.csv").exists():
    from fox_experiments.mechanism.diagnose import diagnose_mechanism
    display(diagnose_mechanism(MECHANISM_OUT)[[
        "recall_lag", "gate", "optimizer", "best_possible_answer_probability",
        "saved_sufficient_radius", "witness_infinite_prefix_radius", "diagnosis"]])

## 8. Optional training-lag sweep

Enable only after short acquisition works. Keep context length and test panels fixed; vary the permitted training lag. Every new acquisition/continuation comparison has its own output folder. These runs are paired within each training range, not across different acquisitions.

In [ ]:
from dataclasses import replace

RUN_R_SWEEP = False
if RUN_R_SWEEP and PROFILE != "smoke":
    for train_lags in ((32, 64), (32, 64, 96, 128)):
        sweep = replace(CONFIG, train_lags=train_lags)
        destination = WORKSPACE / "lag_sweep" / (RUN_NAME + "_R" + str(max(train_lags)))
        run_experiment(DATA, sweep, destination, device=DEVICE, resume=RESUME)
        if (destination / "retrieval_raw.csv").exists():
            summarize_results(destination)
        else:
            print("Acquisition failed:", diagnose_run(destination)["report"])

## 9. Export results

The archive contains the text and controlled-model reports. Datasets and checkpoint weights are excluded; checkpoints remain in the run folders. You can also inspect or download individual CSV files from the Files panel.

For a larger run, use [02_full_training.ipynb](02_full_training.ipynb). Keep every failed acquisition and numerical stop in the analysis; a larger budget is not a guarantee of SGD/Adam separation.

In [ ]:
from fox_experiments.notebook_utils import archive_results

archive = archive_results(WORKSPACE / (RUN_NAME + "_results.zip"),
                          {"text": OUT, "mechanism": MECHANISM_OUT})
print("Saved:", archive)